# This is a Type Validation.
 - Type Validation is the process of checking whether the data provided by a     user matches the expected data type defined in your program.


In [11]:
from pydantic import BaseModel,EmailStr,AnyUrl,Field,field_validator,model_validator
from typing import List,Dict,Optional,Annotated

class Patient(BaseModel):
    name: str
    age: int
    
    weight: float
    married : bool = False
    allergies :Optional[ List[str]]=None
    contact_details: dict[str,str]


def insert_patient_data(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print("inserted into database")

def update_patient_data(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.contact_details)
    print(patient.allergies)
    print("updated")


patient_info = {
    "name": "Saurabh",
    "age": "30",
    "weight":"67",
    
    "contact_details":{"email":'abc@gmail.com', "phone":"8090059781"}
}

patient1 = Patient(**patient_info)


insert_patient_data(patient1)

update_patient_data(patient1)


Saurabh
30
67.0
inserted into database
Saurabh
30
67.0
False
{'email': 'abc@gmail.com', 'phone': '8090059781'}
None
updated


# Data Validation.
 - Data Validation is the process of checking whether input data follows specific rules, conditions, formats, and constraints before your application accepts or processes it.
 
    - For example, in a Patient Management System, we may want to ensure that:

        - Age must be greater than 0 and less than 60.
        - Weight must be a positive number.
        - Email must have a valid format.
        - Name must not exceed 50 characters.
        - Email domain must belong to HDFC or ICICI.
 - Pydantic allows us to implement these rules using BaseModel, Field(), EmailStr, and field_validator.

In [4]:
class Patient(BaseModel):
    name: Annotated[str,Field(max_length=50, title="Name of the patient",description="Give the name of the patient in then 50 chars",examples=["Nitish","Saurabh"])]
    email: EmailStr # Validation email
    age: int = Field(gt=0 , lt = 60)
    linkdin_url: AnyUrl # build in data validation " Any URL "
    weight: Annotated[float, Field(gt=0,strict=False)]
    married : Annotated[bool,Field(default=None,description="Is the patienmt married or not ")]
    allergies :Optional[ List[str]]=None
    contact_details: dict[str,str]


def insert_patient_data(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print("inserted into database")

def update_patient_data(patient: Patient):
    print(patient.name)
    print(patient.email)
    print(patient.linkdin_url)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.contact_details)
    print(patient.allergies)
    print("updated")


patient_info = {
    "name": "Saurabh Kannoujia",
    "age": "59",
    "weight":"67.5",
    "email": "abc@gmail.com",
    "linkdin_url": "http:/linkdin.com/2323",
    "contact_details":{ "phone":"8090059781"}
}

patient1 = Patient(**patient_info)




update_patient_data(patient1)


Saurabh Kannoujia
abc@gmail.com
http://linkdin.com/2323
59
67.5
None
{'phone': '8090059781'}
None
updated


# Field Validator


In [5]:

from pydantic import BaseModel, EmailStr, field_validator


class Patient(BaseModel):

    name: str
    age: int
    email: EmailStr
    weight: float
    married: bool
    allergies: list[str]
    contact_details: dict[str, str]

    @field_validator("email",mode='after')
    @classmethod
    def email_validator(cls, value):

        valid_domains = ["hdfc.com", "icici.com"]

        domain_name = value.split("@")[-1]

        if domain_name not in valid_domains:
            raise ValueError("Not a valid domain")

        return value

    

    @field_validator('name',mode='after')
    @classmethod
    def transform_name(cls, value):
        return value.upper()

    

    @field_validator('age',mode='after')
    @classmethod
    def validate_age(cls,value):
        if 0<value<100:
            return value

        else:
            raise ValueError ("Age should be in between 0 & 100 ")


def updated_patient_details(patient: Patient):

    print(patient.name)
    print(patient.age)
    print(patient.email)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print(patient.contact_details)
    print("Updated")


patient_info = {
    "name": "Saurabh",
    "age": "22",
    "email": "abc@hdfc.com",
    "weight": "56.5",
    "married": True,
    "allergies": ["pollen", "dust"],
    "contact_details": {
        "phone": "8090059781"
    }
}

patient1 = Patient(**patient_info)

updated_patient_details(patient1)



SAURABH
22
abc@hdfc.com
56.5
True
['pollen', 'dust']
{'phone': '8090059781'}
Updated


# Model Validator.
- A Model Validator (@model_validator) is a decorator used to apply custom validation rules to an entire Pydantic model rather than to a single field.
- It is particularly useful when the validation of one field depends on another field.
- For example, in your Patient Management System, suppose you want to implement this condition:
        - If a patient is married, their age must be at least 21.
           - Here, we need to check two fields simultaneously: age and married. This is a suitable use case for a model validator.

In [ ]:
class Patient(BaseModel):
    name : str
    age : int
    email : EmailStr
    weight : float
    married : bool
    allergies : List[str]
    contact_details : dict[str,str]

    @classmethod
    @model_validator(mode="after")
    def validate_emergency_contract(cls, model):
        if model.age >60 and "emergency" not in model.contact_details:
            raise ValueError("Patients older than 60 must have an emergency contact")
        return model


def update_patient_details(patient:Patient):
    print(patient.name)
    print(patient.age)
    print(patient.email)
    print(patient.married)
    print(patient.weight)
    print(patient.allergies)
    print(patient.contact_details)
    print("Updated")


patient_info = {
    "name": "Saurabh kannoujiay",
    "age": "99",
    "email": "abc@gmail.com",
    "weight": "75.5",
    'married': True,
    'allergies': ["dust","smell"],
    "contact_details": {
        "phone":"8090059781",
        "emergency":"8795941835"
    }
}


patient1 = Patient(**patient_info)

update_patient_details(patient1)

    

Saurabh kannoujiay
99
abc@gmail.com
True
75.5
['dust', 'smell']
{'phone': '8090059781', 'emergency': '8795941835'}
Updated


/var/folders/g4/4f2dft7d6d1_s999fn2p83500000gn/T/ipykernel_1132/3244999267.py:11: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.13/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode="after")
